In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
# for dirname, _, filenames in os.walk('/kaggle/input'):
#     for filename in filenames:
#         print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [2]:
!pip install wandb 

In [ ]:
!pip install -U ultralytics

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.0/62.0 kB 4.3 MB/s eta 0:00:00
INFO: pip is looking at multiple versions of mkl-fft to determine which version is compatible with other requirements. This could take a while.
INFO: pip is looking at multiple versions of mkl-random to determine which version is compatible with other requirements. This could take a while.
INFO: pip is looking at multiple versions of mkl-umath to determine which version is compatible with other requirements. This could take a while.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 17.2 MB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16.8/16.8 MB 43.9 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 4.9 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 31.1 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 23.6 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

## Check GPU

In [ ]:
!nvidia-smi

In [ ]:
%%writefile vit.yaml
# ConvNeXt backbone with YOLO head
nc: 12

backbone:
    - [-1, 1, TorchVision, [768, convnext_small, DEFAULT, True, 2, True]]

head:
    - [0, 1, Index, [192, 4]] # P3 features
    - [0, 1, Index, [384, 6]] # P4 features
    - [0, 1, Index, [768, 8]] # P5 features
    - [[1, 2, 3], 1, Detect, [nc]] # Multi-scale detection

### W&B settings

In [ ]:
# !yolo settings wandb=False

In [ ]:
# import wandb
# wandb.login(key="b5f6f8d6d020e78d9cc7abd4b09720b020776968")

In [ ]:
from ultralytics import YOLO

In [ ]:
vit_model = YOLO("vit.yaml",task="detect")

In [ ]:
vit_model.info()

In [ ]:
# vit_model

### Download Data

In [12]:
# Dataset
!pip install roboflow

from roboflow import Roboflow
rf = Roboflow(api_key="pnKvew4NHz0nXfi7EqXH")
project = rf.workspace("tez-enc7o").project("pvelad-uqo5m")
version = project.version(2)
dataset = version.download("yolov12")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 89.9/89.9 kB 3.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 66.8/66.8 kB 5.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 49.9/49.9 MB 35.7 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 60.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.2/4.2 MB 74.4 MB/s eta 0:00:00:00:01
  Attempting uninstall: opencv-python-headless
    Found existing installation: opencv-python-headless 4.12.0.88
    Uninstalling opencv-python-headless-4.12.0.88:
      Successfully uninstalled opencv-python-headless-4.12.0.88
  Attempting uninstall: idna
    Found existing installation: idna 3.10
    Uninstalling idna-3.10:
      Successfully uninstalled idna-3.10
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
bigframes 2.12.0 requires google-cloud-bigquery-st


Extracting Dataset Version Zip to pvelad-2 in yolov12:: 100%|██████████| 16888/16888 [00:03<00:00, 5548.03it/s] 


### Create Augemented data

In [13]:
import os
import cv2
import shutil
import random
import numpy as np
from tqdm import tqdm
import albumentations as A

# CONFIG
img_dir = "/kaggle/working/pvelad-2/train/images"
lbl_dir = "/kaggle/working/pvelad-2/train/labels"
out_img_dir = img_dir  # save augmented images alongside originals
out_lbl_dir = lbl_dir
target_classes = [7, 1, 4, 6]  # classes to augment
target_count = 500
extensions = [".jpg", ".jpeg", ".png"]

# Augmentation pipeline: includes flips + other transforms
aug = A.Compose([
    A.OneOf([
        A.HorizontalFlip(p=1.0),
        A.VerticalFlip(p=1.0),
        A.Compose([A.HorizontalFlip(p=1.0), A.VerticalFlip(p=1.0)]),
    ], p=0.6),
    A.OneOf([
        A.Rotate(limit=15, border_mode=cv2.BORDER_CONSTANT, p=0.4),
        A.ShiftScaleRotate(shift_limit=0.03, scale_limit=0.05, rotate_limit=0, p=0.4),
        A.Affine(shear=10, p=0.2),
    ], p=0.6),
    A.OneOf([
        A.RandomBrightnessContrast(brightness_limit=0.15, contrast_limit=0.15, p=0.6),
        A.HueSaturationValue(hue_shift_limit=10, sat_shift_limit=20, val_shift_limit=10, p=0.6),
    ], p=0.6),
    A.OneOf([
        A.GaussianBlur(blur_limit=5, p=0.2),
        A.GaussNoise(var_limit=(10.0,50.0), p=0.2),
        A.MotionBlur(blur_limit=5, p=0.1)
    ], p=0.25),
], bbox_params=A.BboxParams(format='yolo', label_fields=['class_labels'], min_visibility=0.0))

# Helpers
def list_images(dir):
    return [f for f in os.listdir(dir) if os.path.splitext(f)[1].lower() in extensions]

def read_yolo_labels(lbl_path):
    boxes = []
    classes = []
    if not os.path.exists(lbl_path):
        return boxes, classes
    with open(lbl_path, 'r') as f:
        for line in f.read().strip().splitlines():
            parts = line.split()
            if len(parts) >= 5:
                cls = int(parts[0])
                bbox = list(map(float, parts[1:5]))
                classes.append(cls)
                boxes.append(bbox)
    return boxes, classes

def write_yolo_labels(lbl_path, classes, boxes):
    with open(lbl_path, 'w') as f:
        for c, b in zip(classes, boxes):
            f.write(f"{c} {' '.join(map(str, b))}\n")

# count initial distribution (optional)
counts = {i:0 for i in range(12)}
images = list_images(img_dir)
for img_name in images:
    base = os.path.splitext(img_name)[0]
    boxes, classes = read_yolo_labels(os.path.join(lbl_dir, base + ".txt"))
    for c in classes:
        counts[c] += 1
print("Initial counts:", counts)

# Build per-class image index for selection
class_to_imgs = {c: [] for c in target_classes}
for img_name in images:
    base = os.path.splitext(img_name)[0]
    boxes, classes = read_yolo_labels(os.path.join(lbl_dir, base + ".txt"))
    for c in set(classes):
        if c in target_classes:
            class_to_imgs[c].append(img_name)

# Augmentation loop
augmented_counts = {c: counts.get(c,0) for c in target_classes}
aug_id = 0

for cls in target_classes:
    pbar = tqdm(total=max(0, target_count - augmented_counts[cls]), desc=f"Class {cls}")
    while augmented_counts[cls] < target_count:
        if len(class_to_imgs[cls]) == 0:
            raise RuntimeError(f"No images found containing class {cls} in {img_dir}")
        src_img_name = random.choice(class_to_imgs[cls])
        src_base = os.path.splitext(src_img_name)[0]
        src_img_path = os.path.join(img_dir, src_img_name)
        src_lbl_path = os.path.join(lbl_dir, src_base + ".txt")

        img = cv2.imread(src_img_path)
        if img is None:
            # skip corrupted
            continue
        h, w = img.shape[:2]
        boxes, classes = read_yolo_labels(src_lbl_path)
        if len(boxes) == 0:
            continue

        # apply augmentation
        try:
            augmented = aug(image=img, bboxes=boxes, class_labels=classes)
        except Exception as e:
            # sometimes bbox transforms can fail; skip
            continue
        aug_img = augmented['image']
        aug_boxes = augmented['bboxes']
        aug_classes = augmented['class_labels']

        # discard if target class lost (optional) - ensure augmented image still contains target class
        if cls not in aug_classes:
            # You might still want to keep such augmentations; here we skip
            continue

        # save
        out_base = f"{src_base}_aug{aug_id}"
        out_img_path = os.path.join(out_img_dir, out_base + ".jpg")
        out_lbl_path = os.path.join(out_lbl_dir, out_base + ".txt")
        cv2.imwrite(out_img_path, aug_img)

        # write labels in YOLO format (class x y w h)
        write_yolo_labels(out_lbl_path, aug_classes, aug_boxes)

        aug_id += 1
        # update counts: count each target class presence per image (YOLO counts by image occurrences)
        # If you prefer counting object instances, sum occurrences instead.
        if cls in aug_classes:
            augmented_counts[cls] += 1
            pbar.update(1)
    pbar.close()

print("Final augmented counts (approx):", augmented_counts)


ShiftScaleRotate is a special case of Affine transform. Please use Affine transform instead.
Argument(s) 'var_limit' are not valid for transform GaussNoise


Initial counts: {0: 1650, 1: 9, 2: 2574, 3: 3876, 4: 15, 5: 1674, 6: 48, 7: 3, 8: 648, 9: 282, 10: 2043, 11: 360}


Class 6: 100%|██████████| 452/452 [00:12<00:00, 37.04it/s]

Final augmented counts (approx): {7: 500, 1: 500, 4: 500, 6: 500}


In [14]:
import os, random, glob, shutil
from PIL import Image, ImageEnhance
import math

# CONFIG
train_img_dir = "/kaggle/working/pvelad-2/train/images"
train_lbl_dir = "/kaggle/working/pvelad-2/train/labels"
out_img_dir = "/kaggle/working/train_cp/images"
out_lbl_dir = "/kaggle/working/train_cp/labels"
os.makedirs(out_img_dir, exist_ok=True); os.makedirs(out_lbl_dir, exist_ok=True)
target_classes = {7,1,4,6}            # classes to oversample
copies_per_obj = 10                    # how many synth images per cropped object
max_pastes_per_image = 3
img_ext = ".jpg"
img_size = (1280, 1280)               # if you want to resize outputs

# helpers
def read_label_file(lbl_path):
    items=[]
    with open(lbl_path,'r') as f:
        for line in f.read().strip().splitlines():
            if not line: continue
            c,x,y,w,h = line.split()
            items.append((int(c),float(x),float(y),float(w),float(h)))
    return items

def yolo_to_bbox(item, W,H):
    c,x,y,w,h = item
    cx,cy,wp,hp = x*W, y*H, w*W, h*H
    x1 = int(cx - wp/2); y1 = int(cy - hp/2); x2 = int(cx + wp/2); y2 = int(cy + hp/2)
    return c, (max(0,x1), max(0,y1), min(W,x2), min(H,y2))

def bbox_to_yolo(bbox, W,H, c):
    x1,y1,x2,y2 = bbox
    w = (x2-x1)/W; h = (y2-y1)/H
    x = (x1 + x2) / 2 / W; y = (y1 + y2) / 2 / H
    return f"{c} {x:.6f} {y:.6f} {w:.6f} {h:.6f}\n"

In [15]:
# gather files
img_files = sorted(glob.glob(os.path.join(train_img_dir, "*"+img_ext)))
lbl_files = { os.path.splitext(os.path.basename(p))[0]: p for p in glob.glob(os.path.join(train_lbl_dir,"*.txt")) }

# copy originals to output
for p in img_files:
    name = os.path.basename(p)
    shutil.copy(p, os.path.join(out_img_dir, name))
    key = os.path.splitext(name)[0]
    if key in lbl_files:
        shutil.copy(lbl_files[key], os.path.join(out_lbl_dir, key+".txt"))
    else:
        open(os.path.join(out_lbl_dir, key+".txt"), "w").close()

# extract candidate crops for target classes
crops = []  # tuple: (PIL.Image crop, class)
for img_path in img_files:
    key = os.path.splitext(os.path.basename(img_path))[0]
    lbl_path = lbl_files.get(key)
    if not lbl_path: continue
    img = Image.open(img_path).convert("RGBA")
    W,H = img.size
    labels = read_label_file(lbl_path)
    for item in labels:
        c, bbox = yolo_to_bbox(item, W, H)
        if c in target_classes:
            x1,y1,x2,y2 = bbox
            if x2-x1 < 6 or y2-y1 < 6: continue
            crop = img.crop((x1,y1,x2,y2))
            crops.append((crop, c))

if not crops:
    raise SystemExit("No target-class crops found. Check classes/paths.")

# produce synthesized images
bg_paths = img_files  # reuse existing images as backgrounds
idx = 0
for crop_img, cls in crops:
    for i in range(copies_per_obj):
        bg_path = random.choice(bg_paths)
        bg = Image.open(bg_path).convert("RGBA")
        BW,BH = bg.size

        # don't paste onto image containing same object instance by skipping same filename (optional)
        # random transformations
        scale = random.uniform(0.6, 1.2)
        angle = random.uniform(-25,25)
        nw = max(1, int(crop_img.width * scale))
        nh = max(1, int(crop_img.height * scale))
        rcrop = crop_img.resize((nw,nh), Image.LANCZOS).rotate(angle, expand=True)
        # random position avoiding edges
        px = random.randint(0, max(0, BW - rcrop.width))
        py = random.randint(0, max(0, BH - rcrop.height))

        # paste with alpha mask
        temp = bg.copy()
        temp.paste(rcrop, (px,py), rcrop)

        # optional color jitter
        pil = temp.convert("RGB")
        enhancer = ImageEnhance.Brightness(pil)
        pil = enhancer.enhance(random.uniform(0.8,1.2))

        # save
        out_name = f"cp_{idx:06d}{img_ext}"
        out_path = os.path.join(out_img_dir, out_name)
        pil.save(out_path, quality=95)

        # write label file: start from background labels, add pasted bbox
        base_lbl = []
        base_key = os.path.splitext(os.path.basename(bg_path))[0]
        base_lbl_path = lbl_files.get(base_key)
        if base_lbl_path:
            base_lbl = read_label_file(base_lbl_path)
        # convert base labels to string, but ensure they remain within image after any resize (we didn't resize BG)
        lines = []
        for b in base_lbl:
            c2,x,y,w,h = b
            lines.append(f"{c2} {x:.6f} {y:.6f} {w:.6f} {h:.6f}\n")
        # add pasted bbox (convert px,py to YOLO coords)
        x1,y1 = px,py
        x2,y2 = px + rcrop.width, py + rcrop.height
        lines.append(bbox_to_yolo((x1,y1,x2,y2), BW, BH, cls))
        with open(os.path.join(out_lbl_dir, os.path.splitext(out_name)[0]+".txt"), "w") as f:
            f.writelines(lines)

        idx += 1

print(f"Done. Generated {idx} augmented images in {out_img_dir}")

Done. Generated 750 augmented images in /kaggle/working/train_cp/images


In [16]:
!cp /kaggle/working/train_cp/images/* /kaggle/working/pvelad-2/train/images

In [17]:
!cp /kaggle/working/train_cp/labels/* /kaggle/working/pvelad-2/train/labels

In [14]:
results = vit_model.train(data="/kaggle/working/pvelad-2/data.yaml", 
                          epochs=2, 
                          imgsz=224,
                         device=1)

Ultralytics 8.3.228 🚀 Python-3.11.13 torch-2.6.0+cu124 CUDA:1 (Tesla T4, 15095MiB)
engine/trainer: agnostic_nms=False, amp=True, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/kaggle/working/pvelad-2/data.yaml, degrees=0.0, deterministic=True, device=1, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=2, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=224, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=vit.yaml, momentum=0.937, mosaic=1.0, multi_scale=False, name=train, nbs=64, nms=False, opset=None, optimize=False, optimizer=auto, overlap_mask=True, patience=100, perspective=0.0, plots=True, pose=1

invalid value encountered in less
invalid value encountered in less


                   all        691       1204      0.456      0.255      0.229      0.176
            black_core        149        152      0.767      0.934      0.944      0.798
                corner          1          1          0          0          0          0
                 crack        182        245      0.196     0.0653     0.0737     0.0233
                finger        197        365          0          0          0          0
horizontal_dislocation         51        153      0.217      0.549      0.215     0.0437
               scratch          2          4          1          0          0          0
         short_circuit         52         52          1      0.952      0.981      0.881
            star_crack         20         22      0.379     0.0455     0.0743      0.015
            thick_line        158        198          0          0          0          0
  vertical_dislocation          3         12          1          0          0          0
Speed: 0.0ms preproce

In [19]:
import os

# Define the directory path
directory_path = "/kaggle/working/pvelad-2/test/images"

# Initialize an empty list to store the complete file paths
file_list_with_paths = []

try:
    # Use os.listdir() to get all entries (files and directories) in the path
    all_entries = os.listdir(directory_path)

    # Iterate through the entries
    for entry in all_entries:
        # Construct the complete file path
        full_path = os.path.join(directory_path, entry)

        # Check if the entry is a file
        if os.path.isfile(full_path):
            # Append the complete path to the list
            file_list_with_paths.append(full_path)

    # Print the resulting list of complete file paths
    print(f"Complete file paths found in '{directory_path}':")
    # print(file_list_with_paths)

except FileNotFoundError:
    print(f"Error: The directory '{directory_path}' was not found.")
except Exception as e:
    print(f"An unexpected error occurred: {e}")

Complete file paths found in '/kaggle/working/pvelad-2/test/images':


In [20]:
file_list_with_paths[0:3]

['/kaggle/working/pvelad-2/test/images/img001479_jpg.rf.01cdda32cfa12198f8ed7a965f72f501.jpg',
 '/kaggle/working/pvelad-2/test/images/img001124_jpg.rf.64d7bfd03375a9c607a3502dae880d1e.jpg',
 '/kaggle/working/pvelad-2/test/images/img001133_jpg.rf.8f7c18b4a6f09d1d590d2c74a5f885aa.jpg']

In [21]:
results_batch = vit_model(
   ['/kaggle/working/pvelad-2/test/images/img000323_jpg.rf.eb59d1a1127eb8f6f335bf82a0e3bf2d.jpg',
 '/kaggle/working/pvelad-2/test/images/img002157_jpg.rf.cbdde5648ae107043821ddf755385dfe.jpg',
 '/kaggle/working/pvelad-2/test/images/img000695_jpg.rf.107f8cc2f777fe51e1f3bde177c11af9.jpg']
)  # batch inference


0: 224x224 (no detections), 14.1ms
1: 224x224 (no detections), 14.1ms
2: 224x224 4 cracks, 14.1ms
Speed: 0.8ms preprocess, 14.1ms inference, 0.5ms postprocess per image at shape (1, 3, 224, 224)


In [22]:
!rm -rf /kaggle/working/pvelad-2
print("Data set deleted")

Data set deleted


In [23]:
!zip -r /kaggle/working/train_cp_750.zip /kaggle/working/train_cp

  adding: kaggle/working/train_cp/ (stored 0%)
  adding: kaggle/working/train_cp/labels/ (stored 0%)
  adding: kaggle/working/train_cp/labels/img003525_jpg.rf.921f80bbd1f4629b57c3baadab6257d4.txt (deflated 20%)
  adding: kaggle/working/train_cp/labels/img002055_jpg.rf.7156bac00cc8eb9413d43b4b948ec82f.txt (deflated 20%)
  adding: kaggle/working/train_cp/labels/img002158_jpg.rf.2de4c4e775eda9982633b1e903dbfcd6.txt (deflated 30%)
  adding: kaggle/working/train_cp/labels/img003570_jpg.rf.205eca737de4442edf9ca4edc0debede.txt (deflated 27%)
  adding: kaggle/working/train_cp/labels/img003484_jpg.rf.f6225404356cf16357d4df0d9324176c.txt (deflated 28%)
  adding: kaggle/working/train_cp/labels/img000288_jpg.rf.697403f602fca5191659ebbc9fc917b7.txt (deflated 49%)
  adding: kaggle/working/train_cp/labels/cp_000472.txt (deflated 31%)
  adding: kaggle/working/train_cp/labels/img002567_jpg.rf.0e2768cdefbf49fdd288edd778d45ebd.txt (deflated 54%)
  adding: kaggle/working/train_cp/labels/img002524_jpg.rf.b

In [ ]:
print("Training completed")